## Notebook38

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
klog = pl.read_csv(ub + "data/keylog.csv.gz")
meta = pl.read_csv(ub + "data/keylog-meta.csv.gz")

### Questions

`klog` is a keystroke log of 823 people typing in English, one row per key
press. `t0` and `t1` give the moment of the press and the release, `dur` is
the hold time in between, and `dur_after` is the hold time of the next key
press. `input` is the resulting character, `null` for space and every other
key that produces no character, and `code` is the physical key. `meta`
carries three facts about each of the 823 people behind `id`: `age`, native
`lang`, and `cefr`, a proficiency band for English (`A1/A2` weakest, `C1/C2`
strongest). `t0` and `t1` are milliseconds elapsed since each person's
recording started, not wall-clock times, so they contain no year, month, or
day. Print results unless a question says otherwise.

1. `t0` and `t1` cannot become real dates, since no calendar sits behind
them, but they can still become `Datetime` values. Casting the elapsed
milliseconds to an integer and then to `pl.Datetime("ms")` reads them as a
Unix timestamp, so the clock is anchored at 1970-01-01 instead of at the
moment recording began. Add `press` and `release` columns to `klog` this way,
from `t0` and `t1`, and overwrite `klog` with the result. Then filter to the
one session stored in `klog["id"][0]`, truncate `press` to 10-second bins
with `.dt.truncate("10s")`, group by the bucket, count the keystrokes in each
one, and sort by bucket.

**Answer**:


2. The anchor in question 1 is arbitrary, but it is the *same* arbitrary
moment for all 823 sessions, so a bucket built the same way pools everyone's
first minute together, everyone's second minute together, and so on. Truncate
the full `klog` (not just one session) to 1-minute bins, group by the
bucket, and compute the number of distinct sessions still active and the
average `dur` in each one. Sort by bucket and print the first 15 rows.

**Answer**:


3. Sort `klog` by `id` and `press`, then build a `gap` column: the time
between when the previous key was released and this one was pressed,
`c.press - c.release.shift(1).over(c.id)`. Overwrite `klog` with the result.
Then filter `klog` to rows where `gap` exceeds 2 seconds, a real pause in
typing rather than ordinary key-to-key rhythm, and save it as `pauses`. Group
`pauses` by `id`, count the pauses per person, and print the 5 people with
the most.

The corpus has 38,324 pauses, a median of 41 per person, so the busiest
pauser above (176) is an outlier even among hesitant typists.

4. Compute each person's average `gap` in milliseconds
(`.dt.total_milliseconds()`), join the result onto `meta`, group by `cefr`,
and compute the number of participants and the average gap per group. Sort
by average gap.

**Answer**:


5. `.rolling_mean_by()` smooths a series over a span of time instead of a
count of rows, which suits keystrokes because they are not evenly spaced.
For the session from question 1, compute `gap` in milliseconds and a 15-second
rolling mean of it with `by=c.press`. Plot the raw values in grey and the
smoothed line in red against `press`.

The red line picks out the handful of real hesitations from the grey jitter,
the way the book's rolling mean pulled Chaucer's trajectory out of a jagged
edit history.

6. `.join_asof()` matches each row to the nearest value in another table
instead of an exact one. Build a table of "resume" moments: every row from
question 3's pause filter, keeping only `id` and `press` (renamed
`pause_press`). Asof-join all of `klog` to it, backward, matched by `id`,
and compute `since_pause`, the milliseconds between each keystroke and the
most recent pause that preceded it. Group by whether `since_pause` is under
1000 (recently resumed) and compare the average `dur` and count in each
group, including the group where no pause has happened yet.

**Answer**:
